<a href="https://www.kaggle.com/code/dulapurkaystha/snack-guardian-ai?scriptVersionId=282115657" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Snack Guardian AI: A Multi-Agent Gut-Friendly Snack Assistant

In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


In [2]:
from google.adk.agents import Agent, LlmAgent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types
from google.adk.sessions import DatabaseSessionService
from google.adk.sessions import InMemorySessionService
import json

print("✅ ADK components imported successfully.")

retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

MODEL_NAME = "gemini-2.5-flash-lite"
APP_NAME = "snack_concierge_app"  # Application
USER_ID = "default"  # User
SESSION = "default"  # Session

✅ ADK components imported successfully.


In [3]:
# Define helper functions that will be reused throughout the notebook
async def run_session(
    runner_instance: Runner,
    user_queries: list[str] | str = None,
    session_name: str = "default",
):
    print(f"\n ### Session: {session_name}")

    # Get app name from the Runner
    app_name = runner_instance.app_name

    # Attempt to create a new session or retrieve an existing one
    try:
        session = await session_service.create_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )
    except:
        session = await session_service.get_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )

    # Process queries if provided
    if user_queries:
        # Convert single query to list for uniform processing
        if type(user_queries) == str:
            user_queries = [user_queries]

        # Process each query in the list sequentially
        for query in user_queries:
            print(f"\nUser > {query}")

            # Convert the query string to the ADK Content format
            query_content = types.Content(role="user", parts=[types.Part(text=query)])

            last_text = None
            
            # Stream the agent's response asynchronously
            async for event in runner_instance.run_async(
                user_id=USER_ID, session_id=session.id, new_message=query_content
            ):
                if event.content and event.content.parts:
                    text = event.content.parts[0].text
                    if text and text != "None":
                        last_text = text   # overwritten each sub-agent; last one wins

            if last_text:
                print(f"{MODEL_NAME} > {last_text}")
    else:
        print("No queries!")


print("✅ Helper functions defined.")

✅ Helper functions defined.


## RAG SYSTEM 

In [57]:
# ======================================
# RAG SYSTEM
# ======================================

KNOWLEDGE_BASE = [
    {
        "condition": "GERD",
        "safe": [
            "oatmeal", "banana", "melon", "ginger tea", "fennel tea",
            "rice", "steamed vegetables", "whole grain toast",
            "sweet potato", "cucumber", "pear"
        ],
        "avoid": [
            "chocolate", "caffeine", "coffee", "mint",
            "citrus", "orange", "tomato", "spicy foods",
            "fried foods", "carbonated drinks", "onions",
            "garlic", "alcohol"
        ]
    },

    {
        "condition": "IBS-C",
        "safe": [
            "kiwi", "prunes", "pear", "chia seeds", "ground flax",
            "oatmeal", "warm herbal teas", "lentil soup",
            "stewed apples", "cooked leafy greens"
        ],
        "avoid": [
            "white bread", "excess cheese", "fried foods",
            "heavy meats", "low-fiber snacks", "low-water meals",
            "unripe bananas"
        ]
    },

    {
        "condition": "IBS-D",
        "safe": [
            "white rice", "oatmeal", "banana", "smooth peanut butter",
            "sweet potatoes", "ripe fruit", "white toast",
            "broth soups", "soluble fiber foods"
        ],
        "avoid": [
            "fried foods", "alcohol", "coffee", "high-fat foods",
            "beans", "lactose-heavy foods", "artificial sweeteners",
            "raw cruciferous vegetables", "large salads"
        ]
    },

    {
        "condition": "Lactose Intolerance",
        "safe": [
            "plant-based yogurt", "almond milk", "soy milk",
            "lactose-free milk", "vegan cheese", "oatmeal",
            "rice dishes", "fruits", "vegetables"
        ],
        "avoid": [
            "milk", "ice cream", "soft cheese",
            "whipped cream", "butter (some tolerate ghee)",
            "milk chocolate", "dairy-heavy snacks"
        ]
    },

    {
        "condition": "Crohn's",
        "safe": [
            "white rice", "oatmeal", "bananas", "smooth nut butters",
            "broth soups", "well-cooked vegetables", "rice noodles",
            "mashed potatoes", "avocado", "ripe fruit"
        ],
        "avoid": [
            "popcorn", "nuts", "seeds", "raw vegetables",
            "corn", "fried foods", "high-fiber cereal",
            "beans", "spicy foods"
        ]
    }
]


def gut_condition_lookup(query:str) -> str:
    """
    Consults the medical knowledge base to check if foods are safe or to find soothing foods.
    ARGS:
        query is a string describing the user's condition and/or asking about specific foods.
    
    RETURN:
        A JSON-formatted string representing a list of matched condition entries from the knowledge base. 
        If no match is found, the function returns a simple string:
            "No specific gut dietary data found for this query in the local knowledge base."
    """
    results = []
    q = query.lower()
    for entry in KNOWLEDGE_BASE:
        condition_match = entry["condition"].lower() in q
        safe_match = any(food in q for food in entry["safe"])
        avoid_match = any(food in q for food in entry["avoid"])

        if condition_match or safe_match or avoid_match:
            results.append(entry)

    if not results:
        return "No specific medical dietary data found for this query."
        
    return json.dumps(results, indent=2)

print("Medical RAG Tool complete")

## Condition RAG Agent: Lookup data on a particular gut issue
condition_rag_agent = Agent (
    name="ConditionRAGAgent",
    model=Gemini(
        model=MODEL_NAME,
        retry_options=retry_config
    ),
    instruction="""
    You are a gut-condition lookup agent.

     You have access to the user profile:
    {user_profile_json}

    - If gut_conditions is empty:
        Return the string: None
    
    - If a condition is present:
        Call `medical_rag_tool` once using the user's last message as the query.
        Return EXACTLY what the tool returns (no extra text).
    """,
    tools=[gut_condition_lookup],
    output_key="gut_knowledge",
)

print("✅ Condition RAG Agent defined.")


Medical RAG Tool complete
✅ Condition RAG Agent defined.


## USER PROFILE AGENT

In [58]:
user_profile_agent = Agent(
    name="UserProfileAgent",
    model=Gemini(
        model=MODEL_NAME,
        retry_options=retry_config
    ),
    instruction="""
    Extract user information and return ONLY this JSON object:

    {
      "name": ... or null,
      "diet_preferences": [...],
      "avoid_ingredients": [...],
      "gut_conditions": [...]
    }

    DO NOT speak to the user.
    DO NOT add extra text.
    DO NOT wrap in markdown.
    """,
    output_key="user_profile_json",
)

print("✅ user_profile_agent created.")

✅ user_profile_agent created.


## SNACK CHEF AGENT

In [59]:
snack_chef_agent = Agent(
    name="SnackChefAgent",
    model=Gemini(
        model=MODEL_NAME,
        retry_options=retry_config,
    ),
    instruction="""
    You generate snack ideas ONLY when the user's latest message asks
    for food, snacks, what they can eat, or a recipe.

    If the message does NOT ask for food:
        Return: None

    When generating snacks:
      - Use user_profile_json and gut_knowledge.
      - Respect avoid_ingredients.
      - Create 1–2 snack ideas with 2–4 recipe steps.

    Snacks must NOT be plain fruit or single ingredients.
    They must be *recipes* (something prepared/mixed/cooked).
    
    Output format:
    {
      "snacks": [
        {"name": "...", "recipe": ["step1", "step2"]}
      ]
    }
    """,
    tools=[google_search],
    output_key="snack_suggestions",
)

print("✅ SnackChefAgent created.")


✅ SnackChefAgent created.


## DIALOGUE AGENT

In [60]:
dialogue_agent = Agent(
    name="SnackDialogueAgent",
    model=Gemini(
        model=MODEL_NAME,
        retry_options=retry_config
    ),
    instruction="""
    You are a friendly gut-friendly snack assistant with memory.

    You have access to:
    - user_profile_json: {user_profile_json}
    - gut_knowledge: {gut_knowledge}
    - snack_suggestions: {snack_suggestions}

     Your job:
      - Acknowledge new info (name, diet, gut issues).
      - Summarize what you know when asked.
      - Do NOT show JSON directly.

    Snack rules:
      - Only show snack ideas if the user's CURRENT message asks for food.
      - Only use snack_suggestions when it contains real snacks.
      - If the user didn’t ask for snacks, you may offer:
        “Let me know if you'd like snack ideas.”

    Never break character or mention internal tools.

    Examples of allowed formatting:
    - **bold text**
    - *italics*
    - bullet points
    - simple headings
    """,
)

print("✅ dialogue_agent created.")


✅ dialogue_agent created.


In [61]:
# Root Agent
snack_pipeline_agent = SequentialAgent(
    name="SnackPipelineAgent",
    sub_agents=[
        user_profile_agent,
        condition_rag_agent,
        snack_chef_agent,
        dialogue_agent
    ],
)

root_agent = snack_pipeline_agent

print("✅ root agent created.")


✅ root agent created.


In [81]:
db_url = "sqlite:///my_profile_data.db"
session_service = DatabaseSessionService(db_url=db_url)

print(f"   - Database: my_agent_data.db")

runner = Runner(
    agent=root_agent, 
    app_name=APP_NAME, 
    session_service=session_service,
)

print("✅ Runner created.")

   - Database: my_agent_data.db
✅ Runner created.


In [82]:
await run_session(
    runner, [
        "Hello! I'm Sally and I'm vegetarian.",
        "What do you know about me so far?",
        "I have GERD.",
        "What do you know about my diet and gut issues?",
        "Can you suggest a gentle snack that fits my diet?"
    ],
    "test-sally-01"
)

await run_session(
    runner, [
        "Hello! I'm John and vegetarian.",
        "I do not eat eggs",
        "Can you give me a cake recipe?"
    ],
    "test-john-01"
)


 ### Session: test-sally-01

User > Hello! I'm Sally and I'm vegetarian.
gemini-2.5-flash-lite > Hi Sally! It's great to meet you. I understand you're vegetarian. Let me know if you'd like snack ideas.

User > What do you know about me so far?
gemini-2.5-flash-lite > I know that you are Sally, you prefer a vegetarian diet, and you don't have any specific ingredients you need to avoid or any diagnosed gut conditions.

User > I have GERD.


gemini-2.5-flash-lite > Thanks for letting me know, Sally! I've updated my information. So, you are vegetarian and you have GERD. Let me know if you'd like snack ideas.

User > What do you know about my diet and gut issues?
gemini-2.5-flash-lite > I know that you are Sally, your diet preference is vegetarian, you avoid chocolate, caffeine, coffee, mint, citrus, orange, tomato, spicy foods, fried foods, carbonated drinks, onions, garlic, and alcohol, and you have GERD.

User > Can you suggest a gentle snack that fits my diet?


gemini-2.5-flash-lite > Here are a couple of gentle snack ideas that are vegetarian and suitable for GERD:

**1. Banana Oat Muffins**

These muffins are a great way to enjoy a slightly sweet treat without triggering GERD symptoms. The oats provide fiber, and bananas are generally well-tolerated.

*   **Recipe:**
    1.  Preheat your oven to 350°F (175°C) and grease a muffin tin.
    2.  In a large bowl, combine 1 1/4 cups almond meal, 1/4 cup gluten-free oats, 1 tbsp coconut flour, and 1/4 cup flax seeds.
    3.  In a food processor, blend 2 medium bananas (reserving some slices for topping), 1 tsp cinnamon, 1/2 tsp vanilla extract, 3 tbsp monk fruit sweetener, 1/2 tsp baking soda, 1 tbsp apple cider vinegar, and 1/4 tsp sea salt until well combined.
    4.  Add 5 eggs to the food processor, one at a time, pulsing after each addition until blended.
    5.  Pour the wet ingredients into the dry ingredients and mix until just combined. Fold in pitted and sliced dates.
    6.  Spoon the b

In [80]:
db_path = "my_profile_data.db"

if os.path.exists(db_path):
    os.remove(db_path)
    print("🧹 Deleted my_profile_data.db — all memory cleared.")
else:
    print("No DB file found.")

🧹 Deleted my_profile_data.db — all memory cleared.
